In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import mapclassify
import numpy as np
import optuna
from pathlib import Path
import pandas as pd
import os
import shapely
import rasterio as rio
from glob import glob
from scipy import stats
from scipy.stats import mannwhitneyu
import seaborn as sns
import torch
from torch_geometric.utils import k_hop_subgraph
from torch_sparse import SparseTensor
from tqdm import tqdm
import xarray as xr

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine
from project.utils.unit_conversion import convert_xr_m3s_to_mmd, convert_mmd_to_m3s

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr('/cluster/work/climate/kraftb/streamflow_net/predictions_revision.zarr').sel(
    time=slice('1993', None))

ds['kge'] = compute_metrics(ds.Q_obs_m3s, ds.Q_pred_m3s, metrics=['kge'], dim='time').compute()['kge']

# Load HRU polygons.
hrus = gpd.read_file(
    '/cluster/work/climate/kraftb/data/machflow_routing/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

# rivers = gpd.read_file('/net/argon/landclim/kraftb/machflow_routing/data/raw/EZG_Gewaesser.gpkg', layer='VorfluterAbschnitt', use_arrow=True).set_index('EZGNR')

rivers = gpd.read_file('/cluster/work/climate/kraftb/data/machflow_routing/raw/rivernet/', use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

hrus_with_obs = hrus.loc[ds.hru.where(ds.has_obs.load(), drop=True).values]

bg = load_hillshade(
    hillshade_dir='/cluster/work/climate/kraftb/data/machflow_routing/processed/',
    dem_path='/cluster/work/climate/kraftb/data/machflow_routing/raw/merit.zarr',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    target_crs='EPSG:2056',
    lake_rgb=(86, 150, 245),
    force_recompute=False)

bg_col = load_hillshade(
    hillshade_dir='/cluster/work/climate/kraftb/data/machflow_routing/processed/',
    dem_path='/cluster/work/climate/kraftb/data/machflow_routing/raw/merit.zarr',
    hrus=hrus,
    ds=ds,
    use_terrain=True,
    target_crs='EPSG:2056',
    lake_rgb=(86, 150, 245),
    force_recompute=False)

hru = 145830

In [ ]:
def parametric_return_period(data, dist_name='gumbel_r', fit=True, params=None, max_rp_years=1000):
    """
    Compute return periods in the same order as the input array.
    """
    data = np.asarray(data)
    mask = np.isfinite(data)
    clean_data = data[mask]

    if len(clean_data) == 0:
        return np.full_like(data, np.nan, dtype=float)

    dist = getattr(stats, dist_name)

    if fit:
        params = dist.fit(clean_data)
    elif params is None:
        raise ValueError("You must either set fit=True or provide 'params'.")

    fitted_dist = dist(*params)
    cdf = fitted_dist.cdf(clean_data)

    # Clip to avoid inf
    eps = 1e-12
    cdf = np.clip(cdf, 0, 1 - eps)
    rp_clean = 1.0 / (1.0 - cdf) / 365.25

    # Cap unrealistically large return periods
    rp_clean = np.clip(rp_clean, None, max_rp_years)

    # Put return periods back into original array order
    rp = np.full_like(data, np.nan, dtype=float)
    rp[mask] = rp_clean

    return rp

In [ ]:
ds['Q_pred_rp'] = xr.apply_ufunc(
    parametric_return_period,
    ds.Q_pred_m3s.sel(model='routed_random').chunk(dict(time=len(ds.time))),
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float],
).compute()

In [ ]:
colors = np.vstack((
    # mpl.colormaps['RdPu_r'].resampled(256)(np.linspace(0.3, 0.8, 128)),
    mpl.colormaps['Blues'].resampled(256)(np.linspace(0.3, 1, 128)),
))

cmap_new = mpl.colors.ListedColormap(colors)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.02})

add_aare_outlet = True
add_runoff = False

for i, date in enumerate(['2007-08-09', '2007-08-12']):
    ax = axes[i]

    dss = ds.sel(time=date, model='routed_random')

    if add_runoff:
        cmap = 'Greys'
        alpha = 0.4

        hrus_q = enrich(dss.q_pred_mmd, hrus, 'q')

        bins = [3, 12, 18]
        r = mapclassify.UserDefined(hrus_q.q, bins=bins, lowest=0)
        hrus_q['classes'] = r(hrus_q.q)

        hrus_q.plot(
            column='classes',
            ec='w',
            lw=0.0,
            cmap=cmap,
            alpha=alpha,
            antialiased=True,
            legend=False,
            ax=ax,
            rasterized=True
        )

        legend_text = r.get_legend_classes(fmt="{:.0f}")
        legend_text[-1] = f'> {bins[-1]:.0f}'
        legend_text = [''.join(c for c in s if c not in '[]()').replace('  ', ' ').replace(', ', '-').strip() for s in legend_text]
        num_el = len(legend_text)

        # Get cmap reerence
        colors = mpl.colormaps.get_cmap(cmap)(np.linspace(0, 1, num_el))
        patches = []
        for color in colors:
            patch = mpl.lines.Line2D(
                [0], [0], marker='o', linestyle='None', markeredgewidth=0.3,
                markeredgecolor='k', markerfacecolor=color, markersize=5, alpha=alpha)
            patches.append(patch)

        ax.legend(
            handles=patches,
            labels=legend_text,
            title='Runoff (mm d$^{-1}$)',
            frameon=False,
            fontsize=7,
            title_fontsize=7,
            labelspacing=0.3,
            borderpad=0.2,
            loc='lower right',
            bbox_to_anchor=(1.0, 0.00)
        )

        lakes = hrus_q.loc[ds.hru.where(ds.is_lake.load() & (ds.area.load() > 2), drop=True)]

        lakes.plot(
            ec='k',
            fc='w',
            lw=0.2,
            alpha=1,
            legend=False,
            zorder=30,
            ax=ax,
            rasterized=True
        )
        lakes.plot(
            column='classes',
            ec='w',
            lw=0.0,
            cmap=cmap,
            alpha=alpha,
            legend=False,
            zorder=30,
            ax=ax,
            rasterized=True
        )
    else:
        bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.5, zorder=0)

    hrus_T = enrich(dss.Q_pred_rp, hrus, 'QT')

    rivers_cut = gpd.overlay(
        rivers,
        hrus_T.reset_index()[['EZGNR', 'QT', 'geometry']],
        how='intersection'
    )
    rivers_Q = rivers_cut.explode(index_parts=False).reset_index(drop=True)

    rivers_Q = rivers_Q.sort_values('QT', ascending=[False, True][i])

    rw = np.clip(np.log(rivers_Q.CUM_LEN + 1) / 15 - 0.1, 0.3, None)

    # cmap = sns.diverging_palette(240, 20, as_cmap=True)
    # cmap = cmap_new
    cmap = 'coolwarm'

    # norm = mpl.colors.LogNorm(vmin=0.1, vmax=3000)
    norm = mpl.colors.LogNorm(vmin=0.01, vmax=100)

    rivers_Q.plot(
        color='0.3',
        linewidth=rw + 0.2,
        zorder=2,
        capstyle='round',
        legend=False,
        ax=ax)

    rivers_Q.plot(
        column='QT',
        linewidth=rw - 0.3,
        norm=norm,
        zorder=2,
        capstyle='round',
        fc='none',
        alpha=1.0,
        cmap=cmap,
        legend=False,
        ax=ax)

    axins = ax.inset_axes((0.03, 0.9, 0.25, 0.02), zorder=25)
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(
        mappable,
        cax=axins,
        orientation='horizontal',   # or 'vertical'
        extend='max',
        label='Discharge return\nperiod (years)'
    )
    cbar.ax.xaxis.label.set_size(7)
    cbar.ax.tick_params(labelsize=7)

    ax.text(0.01, 0.99, f'Peak flood {date}' if i == 0 else f'Post flood {date}', fontsize=8, transform=ax.transAxes,
        verticalalignment='top', horizontalalignment='left',
        zorder=40)

    if add_aare_outlet:
        x = hrus.loc[hru].geometry.centroid.x
        y = hrus.loc[hru].geometry.centroid.y

        ax.scatter(x, y, ec='k', fc='tab:red', s=12, lw=0.8, zorder=9999)

    ax.set_title('')

if add_runoff:
    postfix = ''
else:
    postfix = '_norunoff'

if add_aare_outlet:
    postfix += '_aare_outlet'
else:
    postfix += ''

save_path = f'./figures/discharge_map_plain{postfix}.png'

fig.savefig(save_path, dpi=300, bbox_inches='tight')


In [ ]:
ds

In [ ]:
def plot_ts(ds, ax, time_start, time_end, pred_labels):
    ds_s = ds.sel(time=slice(time_start, time_end))
    ds_s.Q_obs_m3s.sel(
        time=slice(time_start, time_end)).plot(
            ax=ax,
            color='0.4',
            lw=1.8,
            label='Observed'
        )

    for i, model in enumerate(['routed_random', 'instant_random', 'lumped_random']):
        color = ["#DAA507", "#FF6347", "#1E88E5"][i]

        ds_s.Q_pred_m3s.sel(model=model).plot(
            ax=ax,
            label=pred_labels[i],
            alpha=1.0,
            lw=0.9,
            ls='-',
            # dashes=[3, 1],
            color=color,
            zorder=5-i
        )


fig = plt.figure(figsize=(10, 5))

gs = mpl.gridspec.GridSpec(2, 2, figure=fig, wspace=0.1, hspace=0.25)

ds_aare = ds.sel(hru=145830)

ax_tl = fig.add_subplot(gs[0, 0])
ax_tr = fig.add_subplot(gs[0, 1], sharey=ax_tl)
ax_b = fig.add_subplot(gs[1, :])

z0 = ('2006-01-01', '2008-12-31')
z1 = ('2006-07-01', '2006-10-31')
z2 = ('2007-07-01', '2007-10-31')

routed_random_kge = ds_aare.kge.sel(model='routed_random').item()
routed_instant_kge = ds_aare.kge.sel(model='instant_random').item()
lumped_random_kge = ds_aare.kge.sel(model='lumped_random').item()

labels = [
    f'DROP (routed; KGE={routed_random_kge:.2f})',
    f'DROP (instant; KGE={routed_instant_kge:.2f})',
    f'LSTM (lumped; KGE={lumped_random_kge:.2f})'
]

plot_ts(ds_aare, ax_b, *z0, labels)
plot_ts(ds_aare, ax_tl, *z1, labels)
plot_ts(ds_aare, ax_tr, *z2, labels)

# for i, model in enumerate(['routed_random', 'lumped_random']):
#     color = ['tab:red', 'tab:blue'][i]
#     Q_pred_m3s.sel(model=model).plot(ax=ax_b, label=model.replace('_', ' ').title(), lw=0.7, ls='--', alpha=0.8, color=color)

# Add the connection patches

for s, (zoom_start, zoom_end) in enumerate([z1, z2]):
    top_axis = [ax_tl, ax_tr][s]

    zoom_start = pd.Timestamp(zoom_start)
    zoom_end = pd.Timestamp(zoom_end)

    ax_b.axvspan(zoom_start, zoom_end, ec='k', fc='0.92', lw=0.4, ls='--', zorder=-2)

    for z, zoom in enumerate([zoom_start, zoom_end]):

        fig.add_artist(mpl.patches.ConnectionPatch(
            xyA=(z, 0), coordsA=top_axis.transAxes,
            xyB=(mpl.dates.date2num(zoom), ax_b.get_ylim()[1]), coordsB=ax_b.transData,
            color='k',
            lw=0.4,
            ls='--',
            zorder=0,
        ))

for i, ax in enumerate([ax_tl, ax_tr, ax_b]):
    ax.set_xlabel('')
    despine(ax)
    ax.set_title('')

    if i != 1:
        ax.set_ylabel('Discharge (m$^3$ s$^{-1}$)')
    else:
        ax.set_ylabel('')

ax_b.legend(loc='upper right', bbox_to_anchor=(1, 1), frameon=False)

for i, ax in enumerate([ax_tr, ax_b]):
    event = pd.Timestamp('2007-08-09')
    if i == 0:
        post_event = pd.Timestamp('2007-08-13')
    else:
        post_event = pd.Timestamp('2007-08-19')

    # ax.axvline(mpl.dates.date2num(event), color='k', lw=0.6, ls='--', zorder=-1)
    ax.text(
        mpl.dates.date2num(post_event),
        ax.get_ylim()[1] * 0.9,
        'Peak flood event\n9 August 2007',
        color='k',
        fontsize=6,
        rotation=90,
        va='top',
        ha='left',
        path_effects=[mpe.withStroke(linewidth=1, foreground="w")]
    )

# date_form = mpl.dates.DateFormatter("%m-%d")
ax_tl.xaxis.set_major_locator(mpl.dates.MonthLocator(bymonthday=1))
ax_tr.xaxis.set_major_locator(mpl.dates.MonthLocator(bymonthday=1))

fig.align_ylabels([ax_tl, ax_b])

fig.savefig('./figures/time_series.pdf', dpi=300, bbox_inches='tight')

In [ ]:
def enrich_with_grads(time_idx: int, var: str, agg_fn: str, consider_n_past: int = 7) -> tuple[gpd.GeoDataFrame, str]:
    time_start = ds_s.grad_time.isel(grad_time=max(time_idx - consider_n_past, 0)).dt.strftime('%Y-%m-%d').item()
    time_end = ds_s.grad_time.isel(grad_time=time_idx).dt.strftime('%Y-%m-%d').item()
    if 'grad_time' not in ds_s[var].dims:
        ds_subset = ds_s[var].sel(time=slice(time_start, time_end))
    else:
        ds_subset = ds_s[var].isel(grad_time=time_idx)

    if agg_fn == 'mean':
        ds_subset = ds_subset.mean('time')
    elif agg_fn == 'sum':
        ds_subset = ds_subset.sum('time')
    hrus_attr = enrich(
        ds=ds_subset,
        hrus=hrus.loc[ds_s.hru],
        variables=var
    )

    return hrus_attr, time_end

def select_dayofyear_window(ds: xr.Dataset | xr.DataArray, date: str, window: int) -> xr.Dataset | xr.DataArray:
    center_doy = ds.sel(time=date).time.dt.dayofyear.item()
    query_days = np.arange(center_doy - window, center_doy + window + 1)
    return ds.sel(time=ds.time.dt.dayofyear.isin(query_days))


In [ ]:
dss = xr.open_zarr('/cluster/work/climate/kraftb/streamflow_net/routed/random/tune/source_contrib_145830.zarr/')
ds_aare = ds.sel(hru=hru)
source_ref = select_dayofyear_window(dss.q_source_m3s, date='2007-08-09', window=10).mean('time').compute()
source_day = dss.q_source_m3s.sel(time='2007-08-09').compute()
source_diff = source_day - source_ref
source = xr.Dataset({
    'ref_m3s': source_ref,
    'day_m3s': source_day,
    'diff_m3s': source_diff,
    'area': dss.area,
})

convert_xr_m3s_to_mmd(source, variables=['ref_m3s', 'day_m3s', 'diff_m3s']).compute()

In [ ]:
hrus_source = enrich(
    source.day_mmd.sel(lag=0),
    hrus.loc[source.hru],
    variables='lag_instant')
hrus_source = enrich(
    source.day_mmd.sel(lag=slice(3, 1)).sum('lag'),
    hrus_source,
    variables='lag_1w')
hrus_source = enrich(
    source.day_mmd.sel(lag=slice(30, 4)).sum('lag'),
    hrus_source, variables='lag_remaining')

hrus_source = enrich(
    ds.prec.sel(time='2007-08-09', hru=hrus_source.index),
    hrus_source,
    variables='prec_instant')
hrus_source = enrich(
    ds.prec.sel(time=slice('2007-08-06', '2007-08-08'), hru=hrus_source.index).mean('time'),
    hrus_source,
    variables='prec_1w')
hrus_source = enrich(
    ds.prec.sel(time=slice('2007-07-10', '2007-08-05'), hru=hrus_source.index).mean('time'),
    hrus_source,
    variables='prec_remaining')

hrus_source = enrich(
    ds.q_pred_mmd.sel(
        time='2007-08-09', hru=hrus_source.index, model='routed_random'),
    hrus_source,
    variables='q_instant')
hrus_source = enrich(
    ds.q_pred_mmd.sel(
        time=slice('2007-08-06', '2007-08-08'), model='routed_random', hru=hrus_source.index).mean('time'),
    hrus_source,
    variables='q_1w')
hrus_source = enrich(
    ds.q_pred_mmd.sel(
        time=slice('2007-07-10', '2007-08-05'), model='routed_random', hru=hrus_source.index).mean('time'),
    hrus_source,
    variables='q_remaining')

hrus_source = enrich(source.area, hrus_source, variables='area')

In [ ]:
variables = [
    ['prec_remaining', 'prec_1w', 'prec_instant'],
    ['q_remaining', 'q_1w', 'q_instant'],
    ['lag_remaining', 'lag_1w', 'lag_instant'],
]
titles = [
    'July 10 to August 5 (lag 4 to 31 days)',
    'August 6-8 (lag 1 to 3 days)',
    'August 9 (instant)',
]
ylabels = [
    'Observed\nprecipitation (mean)',
    'Simulated\nrunoff (mean)',
    'Contribution to\nAugust 9 2007',
]

In [ ]:
xmin, ymin, xmax, ymax = hrus_source.total_bounds
xrng = xmax - xmin
xmin -= xrng * 0.05
xmax += xrng * 0.05
yrng = ymax - ymin
ymin -= yrng * 0.05
ymax += yrng * 0.05
bounds = [xmin, xmax, ymin, ymax]

bg_bounds = bg.sel(lon=slice(*bounds[:2]), lat=slice(bounds[-1], bounds[-2])).load()

In [ ]:
fig = plt.figure(layout='constrained', figsize=(8, 4))

gs = GridSpec(3, 4, figure=fig, wspace=0.02, hspace=0.0, width_ratios=[1, 1, 1, 0.02])

total_q_m3s = convert_mmd_to_m3s(source.day_mmd.sum('lag').values, hrus_source['area'].values).sum()

for r in range(3):
    vmax = np.quantile(hrus_source[variables[r]], 0.98)
    if r in (0, 1):
        vmax = 45

    norm = mpl.colors.Normalize(vmin=0, vmax=vmax)

    kwargs = {
        'cmap': ['GnBu', 'GnBu', 'PuRd'][r],
        'ec': 'k',
        'lw': 0.1,
        'alpha': 0.7,
        'legend': False,
    }

    for c in range(3):

        ax = fig.add_subplot(gs[r, c], projection=lv95)

        x = hrus_source.loc[hru].geometry.centroid.x
        y = hrus_source.loc[hru].geometry.centroid.y

        ax.set_extent(bounds, crs=lv95)
        # bg_bounds.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)

        rivers.loc[rivers.CUM_LEN > 20000].plot(ax=ax, color='k', lw=0.5, zorder=20, alpha=0.7)

        var = variables[r][c]

        hrus_source.plot(
            column=var,
            ax=ax,
            norm=norm,
            rasterized=True,
            **kwargs
        )

        if c == 0:
            ax.text(-0.1, 0.5, ylabels[r], rotation=90, va='center', ha='center', transform=ax.transAxes, fontsize=8)
        
        if r == 0:
            ax.set_title(titles[c], fontsize=8)
        else:
            ax.set_title('')

        if r == 2:
            contr = convert_mmd_to_m3s(hrus_source[var].values, hrus_source['area'].values).sum()
            contr_perc = contr / total_q_m3s * 100
            ax.text(
                0.015, 0.98, f'Total contribution:\n{contr:.0f} m$^3$ s$^{{-1}}$ ({contr_perc:.0f} %)', transform=ax.transAxes,
                ha='left', va='top', fontsize=7, bbox=dict(facecolor='0.93', alpha=1, edgecolor='none', pad=1), zorder=100)

        ax.scatter(x, y, ec='k', fc='tab:red', s=12, lw=0.6, zorder=9999)

    cbar_ax = fig.add_subplot(gs[r, -1])
    cax = cbar_ax.inset_axes([0.0, 0.1, 1.0, 0.8])
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=kwargs['cmap'])
    cbar = plt.colorbar(
        mappable,
        cax=cax,
        orientation='vertical',
        extend='max',
        label=['Precipitation (mm d$^{-1}$)', 'Runoff (mm d$^{-1}$)', 'Contribution (mm d$^{-1}$)'][r],
    )
    cbar.ax.yaxis.label.set_size(8)
    cbar.ax.tick_params(labelsize=8)
    cbar_ax.axis('off')

fig.savefig('./figures/case_study.png', dpi=300, bbox_inches='tight')

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(10, 4), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.02})

# for i, date in enumerate(['2007-08-09', '2007-08-12']):
#     ax = axes[i]

#     dss = ds.sel(time=date, model='routed_random')

#     hrus_q = enrich(dss.q_pred_mmd, hrus, 'q')
#     hrus_Q = enrich(dss.Q_pred_m3s, hrus, 'Q')

#     # rivers_Q = gpd.sjoin(
#     #     rivers,               # left (the geometry you want to keep)
#     #     hrus_Q.reset_index()[['EZGNR', 'Q', 'geometry']],   # right (the attribute you want to bring over)
#     #     how="left",          # keep all lines; unmatched lines get NaN
#     #     predicate="intersects"
#     # ).sort_values('CUM_LEN', ascending=True)
#     # rivers_Q = rivers_Q.loc[rivers_Q.EZGNR.notna()]

#     rivers_cut = gpd.overlay(
#         rivers,
#         hrus_Q.reset_index()[['EZGNR', 'Q', 'geometry']],
#         how='intersection'
#     )
#     rivers_Q = rivers_cut.explode(index_parts=False).reset_index(drop=True)

#     rivers_Q = rivers_Q.sort_values('Q', ascending=True)

#     rw = np.clip(np.log(rivers_Q.CUM_LEN + 1) / 15 - 0.1, 0.3, None)

#     ax.set_global()
#     ax.set(zorder=1000)

#     cmap = 'inferno'
#     norm = mpl.colors.LogNorm(vmin=1, vmax=50)
#     alpha = 0.5

#     hrus_q.plot(
#         column='q',
#         ec='k',
#         lw=0.05,
#         cmap=cmap,
#         alpha=alpha,
#         antialiased=True,
#         norm=norm,
#         ax=ax
#     )

#     axins = ax.inset_axes((0.74, 0.16, 0.25, 0.02), zorder=30)
#     mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
#     cbar = plt.colorbar(
#         mappable,
#         cax=axins,
#         alpha=alpha,
#         orientation='horizontal',   # or 'vertical'
#         extend='max',
#         label='Runoff (log, mm d$^{-1}$)'
#     )
#     cbar.ax.xaxis.label.set_size(7)
#     cbar.ax.tick_params(labelsize=7, which='both')


#     cmap = 'YlGnBu_r'
#     norm = mpl.colors.LogNorm(vmin=0.1, vmax=3000)

#     rivers_Q.plot(
#         color='k',
#         linewidth=rw + 0.5,
#         zorder=2,# 
#         capstyle='round',
#         legend=False,
#         ax=ax)


#     rivers_Q.plot(
#         column='Q',
#         linewidth=rw + 0.1,
#         norm=norm,
#         zorder=2,
#         capstyle='round',
#         fc='none',
#         cmap=cmap,
#         legend=False,
#         ax=ax)

#     axins = ax.inset_axes((0.06, 0.92, 0.25, 0.02), zorder=25)
#     mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
#     cbar = plt.colorbar(
#         mappable,
#         cax=axins,
#         orientation='horizontal',   # or 'vertical'
#         extend='max',
#         label='Discharge (log, m$^3$ s$^{-1}$)'
#     )
#     cbar.ax.xaxis.label.set_size(7)
#     cbar.ax.tick_params(labelsize=7)

#     hrus_q.loc[ds.hru.where(ds.is_lake.load(), drop=True)].plot(
#         fc='powderblue',
#         ec='k',
#         lw=0.2,
#         ax=ax,
#         zorder=30
#     )

#     ax.text(0.01, 0.99, f'Peak flood {date}' if i == 0 else f'Post flood {date}', fontsize=8, transform=ax.transAxes,
#         verticalalignment='top', horizontalalignment='left',
#         zorder=40)

#     x = hrus_source.loc[hru].geometry.centroid.x
#     y = hrus_source.loc[hru].geometry.centroid.y

#     ax.scatter(x, y, ec='k', fc='tab:red', s=12, lw=0.6, zorder=9999)

# fig.savefig('./figures/discharge_map.png', dpi=300, bbox_inches='tight')

In [ ]:
def plpt_ts(ds, ax, time_start, time_end, pred_labels):
    ds_s = ds.sel(time=slice(time_start, time_end))
    ds_s.Q_obs_m3s.sel(
        time=slice(time_start, time_end)).plot(
            ax=ax,
            color='0.4',
            lw=1.4,
            label='Observed'
        )

    for i, model in enumerate(['routed_random', 'lumped_random']):
        color = ["#DAA507", "#1E88E5"][i]

        ds_s.Q_pred_m3s.sel(model=model).plot(
            ax=ax,
            label=pred_labels[i],
            alpha=0.9,
            lw=1.,
            ls='-',
            # dashes=[3, 1],
            color=color,
            zorder=5-i
        )


fig = plt.figure(figsize=(10, 5))

gs = mpl.gridspec.GridSpec(2, 2, figure=fig, wspace=0.1, hspace=0.25)

ds_rhone = ds.sel(hru=141936)

ax_tl = fig.add_subplot(gs[0, 0])
ax_tr = fig.add_subplot(gs[0, 1], sharey=ax_tl)
ax_b = fig.add_subplot(gs[1, :])

z0 = ('2006-01-01', '2008-12-31')
z1 = ('2006-05-01', '2006-12-31')
z2 = ('2007-05-01', '2007-12-31')

routed_random_kge = ds_rhone.kge.sel(model='routed_random').item()
lumped_random_kge = ds_rhone.kge.sel(model='lumped_random').item()

labels = [
    f'DRAIN (routed; KGE={routed_random_kge:.2f})',
    f'LSTM (lumped; KGE={lumped_random_kge:.2f})'
]

plpt_ts(ds_rhone, ax_b, *z0, labels)
plpt_ts(ds_rhone, ax_tl, *z1, labels)
plpt_ts(ds_rhone, ax_tr, *z2, labels)

# for i, model in enumerate(['routed_random', 'lumped_random']):
#     color = ['tab:red', 'tab:blue'][i]
#     Q_pred_m3s.sel(model=model).plot(ax=ax_b, label=model.replace('_', ' ').title(), lw=0.7, ls='--', alpha=0.8, color=color)

# Add the connection patches

for s, (zoom_start, zoom_end) in enumerate([z1, z2]):
    top_axis = [ax_tl, ax_tr][s]

    zoom_start = pd.Timestamp(zoom_start)
    zoom_end = pd.Timestamp(zoom_end)

    ax_b.axvspan(zoom_start, zoom_end, ec='k', fc='0.92', lw=0.4, ls='--')

    for z, zoom in enumerate([zoom_start, zoom_end]):

        fig.add_artist(mpl.patches.ConnectionPatch(
            xyA=(z, 0), coordsA=top_axis.transAxes,
            xyB=(mpl.dates.date2num(zoom), ax_b.get_ylim()[1]), coordsB=ax_b.transData,
            color='k',
            lw=0.4,
            ls='--',
            zorder=0,
        ))

for i, ax in enumerate([ax_tl, ax_tr, ax_b]):
    ax.set_xlabel('')
    despine(ax)
    ax.set_title('')

    if i != 1:
        ax.set_ylabel('Discharge (m$^3$ s$^{-1}$)')
    else:
        ax.set_ylabel('')

ax_b.legend(loc='upper right', bbox_to_anchor=(1, 1), frameon=False)

for i, ax in enumerate([ax_tr, ax_b]):
    event = pd.Timestamp('2007-08-09')
    if i == 0:
        post_event = pd.Timestamp('2007-08-14')
    else:
        post_event = pd.Timestamp('2007-08-19')

    ax.axvline(mpl.dates.date2num(event), color='k', lw=0.6, ls='--')
    ax.text(
        mpl.dates.date2num(post_event),
        ax.get_ylim()[1] * 0.9,
        'Peak flood event',
        color='k',
        fontsize=6,
        rotation=90,
        va='top',
        ha='left',
        path_effects=[mpe.withStroke(linewidth=1, foreground="w")]
    )

fig.align_ylabels([ax_tl, ax_b])

fig.savefig('./figures/rhone_ID2009_hydropeaking.png', dpi=300, bbox_inches='tight')